In [2]:
import os
import torch
import xml.etree.ElementTree as ET
import numpy as np
import cv2
import torchvision.transforms as transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import torchvision
from torch.utils.data import random_split

# Class names (ensure order matches dataset)
CLASS_NAMES = [
    "black_core", "corner", "crack", "finger", "fragment", "horizontal_dislocation",
    "printing_error", "scratch", "short_circuit", "star_crack", "thick_line", "vertical_dislocation"
]

# Mapping class names to indices
CLASS_MAP = {name: idx + 1 for idx, name in enumerate(CLASS_NAMES)}  # Background class is 0

# Data transforms
transform = A.Compose([
    A.Resize(1024, 1024),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    ToTensorV2()
])

class SolarDefectDataset(Dataset):
    def __init__(self, img_dir, xml_dir, transforms=None):
        self.img_dir = img_dir
        self.xml_dir = xml_dir
        self.transforms = transforms
        self.img_files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_filename = self.img_files[idx]
        img_path = os.path.join(self.img_dir, img_filename)
        xml_path = os.path.join(self.xml_dir, img_filename.replace('.jpg', '.xml'))

        # Read image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Parse XML annotation
        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall('object'):
            name = obj.find('name').text
            bbox = obj.find('bndbox')

            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_MAP[name])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self.transforms:
            augmented = self.transforms(image=img)
            img = augmented["image"].float() / 255.0  # Convert to float & normalize

        return img, target

# Directories
IMG_DIR = "D:/projects/solar_defect/dataset/trainval/JPEGImages"
XML_DIR = "D:/projects/solar_defect/dataset/trainval/Annotations"

# Create dataset
full_dataset = SolarDefectDataset(IMG_DIR, XML_DIR, transform)

# Split into train (80%) and validation (20%)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])


train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


C:\Users\jayve\anaconda3\envs\abc\Lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.5 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [3]:
import torchvision.models.detection as detection

# Load Faster R-CNN Model with ResNet-50 backbone
def get_model(num_classes):
    model = detection.fasterrcnn_resnet50_fpn(weights="COCO_V1")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    return model

# Create model with our defect classes
num_classes = len(CLASS_NAMES) + 1  # Add background class (0)
model = get_model(num_classes)
model = model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))


In [4]:
def iou(box1, box2):
    """
    Compute Intersection over Union (IoU) for two bounding boxes.
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0


In [5]:
import torch.optim as optim
from tqdm import tqdm
import torch
from torchvision.ops import box_iou
import torch
from torchvision.ops import box_iou

# Training parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 20
debug = False  # Set True to run a small dataset test

# Split dataset into training and validation sets
train_size = int(0.8 * len(full_dataset))  # 80% for training
val_size = len(full_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Debug mode: Use smaller datasets for quick testing
if debug:
    train_dataset = torch.utils.data.Subset(train_dataset, range(10))  # Small train set
    val_dataset = torch.utils.data.Subset(val_dataset, range(5))  # Small val set

# Create separate DataLoaders for training & validation
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Training function

# Training function with separate validation
def train_one_epoch(model, train_dataloader, val_dataloader, optimizer):
    model.train()  # Set model to training mode
    total_train_loss = 0

    # Training Loop
    for images, targets in tqdm(train_dataloader, desc="Training"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)  # Forward pass
        train_loss = sum(loss for loss in loss_dict.values())

        train_loss.backward()
        optimizer.step()
        total_train_loss += train_loss.item()

    # Compute average training loss
    avg_train_loss = total_train_loss / len(train_dataloader)

    # Run validation
    val_loss, accuracy = validate(model, val_dataloader)

    return avg_train_loss, val_loss, accuracy


@torch.no_grad()
def validate(model, val_dataloader):
    """Validates the model by computing loss (in train mode) and accuracy (in eval mode)."""
    
    model.train()  # ✅ Keep model in training mode for loss computation
    total_val_loss = 0.0
    total_correct = 0
    total_samples = 0

    # 🔹 Compute Validation Loss (Using model.train())
    with torch.no_grad():
        for images, targets in tqdm(val_dataloader, desc="Validating", leave=False):
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            val_loss_dict = model(images, targets)  # Forward pass to get loss

            if isinstance(val_loss_dict, dict):
                val_loss = sum(v for v in val_loss_dict.values())  
            else:
                raise ValueError(f"Unexpected val_loss_dict format: {type(val_loss_dict)} -> {val_loss_dict}")

            total_val_loss += val_loss.item()

    # 🔹 Compute Validation Accuracy (Switch to eval mode)
    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(val_dataloader, desc="Computing Accuracy", leave=False):
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            model_outputs = model(images)

            for i in range(len(targets)):
                pred_boxes = model_outputs[i]['boxes'].detach().cpu()
                pred_labels = model_outputs[i]['labels'].detach().cpu()
                
                true_boxes = targets[i]['boxes'].detach().cpu()
                true_labels = targets[i]['labels'].detach().cpu()

                total_samples += len(true_boxes)  # Total ground-truth boxes

                if len(pred_boxes) > 0 and len(true_boxes) > 0:
                    iou_matrix = box_iou(pred_boxes, true_boxes)  # Compute IoU matrix
                    
                    # 🔹 Match each ground truth box to the best prediction
                    for gt_idx in range(len(true_boxes)):
                        max_iou, best_pred_idx = iou_matrix[:, gt_idx].max(dim=0)

                        if max_iou > 0.5 and pred_labels[best_pred_idx] == true_labels[gt_idx]:
                            total_correct += 1  # Count as correct match

    # 🔹 Compute Metrics
    avg_val_loss = total_val_loss / len(val_dataloader) if len(val_dataloader) > 0 else 0.0
    accuracy = (total_correct / total_samples) * 100 if total_samples > 0 else 0.0
    accuracy = min(accuracy, 100)  # Prevent accuracy > 100%

    return avg_val_loss, accuracy

# Training Loop
num_epochs = 20

for epoch in range(num_epochs):
    train_loss, val_loss, accuracy = train_one_epoch(model, train_dataloader, val_dataloader, optimizer)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Accuracy: {accuracy:.4f}")

# Save trained model
torch.save(model.state_dict(), "solar_defect_trained_model.pth")
print("Model training completed and saved!")


Training: 100%|██████████████████████████████████████████████████████████████████████| 900/900 [22:38<00:00,  1.51s/it]


Epoch [1/20] - Train Loss: 0.2829 - Val Loss: 0.2186 - Accuracy: 61.2801


Training: 100%|██████████████████████████████████████████████████████████████████████| 900/900 [22:14<00:00,  1.48s/it]


Epoch [2/20] - Train Loss: 0.2127 - Val Loss: 0.1979 - Accuracy: 61.2801


Training: 100%|██████████████████████████████████████████████████████████████████████| 900/900 [22:13<00:00,  1.48s/it]


...

Epoch [18/20] - Train Loss: 0.1143 - Val Loss: 0.1857 - Accuracy: 74.7782


Training: 100%|██████████████████████████████████████████████████████████████████████| 900/900 [22:05<00:00,  1.47s/it]


Epoch [19/20] - Train Loss: 0.1132 - Val Loss: 0.1863 - Accuracy: 74.7782


Training: 100%|██████████████████████████████████████████████████████████████████████| 900/900 [22:05<00:00,  1.47s/it]


Epoch [20/20] - Train Loss: 0.1106 - Val Loss: 0.1799 - Accuracy: 73.8910
Model training completed and saved!


In [7]:
import torch
import torchvision.models.detection as detection

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load trained model
def load_trained_model(model_path, num_classes):
    print("Initializing Faster R-CNN model...")
    model = detection.fasterrcnn_resnet50_fpn(weights=None)  # No pre-trained weights
    
    # Modify classifier head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    print("Model architecture modified for defect classification.")

    # Load trained weights
    print(f"Loading model weights from: {model_path}")
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Model weights loaded successfully.")

    # Move to device
    model.to(device)
    print("Model moved to device.")

    # Set to evaluation mode
    model.eval()
    print("Model set to evaluation mode.")

    return model

# Define class names
CLASS_NAMES = [
    "black_core", "corner", "crack", "finger", "fragment", "horizontal_dislocation",
    "printing_error", "scratch", "short_circuit", "star_crack", "thick_line", "vertical_dislocation"
]

# Define number of classes
num_classes = len(CLASS_NAMES) + 1  
print(f"Number of classes (including background): {num_classes}")

# Define model path
model_path = "C:\\Users\\jayve\\solar_defect_trained_model.pth"
print(f"Model path: {model_path}")

# Load the trained model
model = load_trained_model(model_path, num_classes)
print("Model successfully loaded and ready for inference.")


Using device: cuda
Number of classes (including background): 13
Model path: C:\Users\jayve\solar_defect_trained_model.pth
Initializing Faster R-CNN model...


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\jayve/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth
100%|█████████████████████████████████| 97.8M/97.8M [00:29<00:00, 3.48MB/s]


Model architecture modified for defect classification.
Loading model weights from: C:\Users\jayve\solar_defect_trained_model.pth
Model weights loaded successfully.
Model moved to device.
Model set to evaluation mode.
Model successfully loaded and ready for inference.


In [ ]:
import torch
import torchvision.models.detection as detection
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as T
import cv2
import os
import matplotlib.pyplot as plt

# Set device
device = torch.device("cpu")
print(device)
# Define class names
CLASS_NAMES = [
    "black_core", "corner", "crack", "finger", "fragment", "horizontal_dislocation",
    "printing_error", "scratch", "short_circuit", "star_crack", "thick_line", "vertical_dislocation"
]
num_classes = len(CLASS_NAMES) + 1  # Add background class

# Load trained Faster R-CNN model
def load_trained_model(model_path, num_classes):
    print("Loading Faster R-CNN model...")

    model = detection.fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    print(f"Loading trained weights from: {model_path}")
    model.load_state_dict(torch.load(model_path, map_location=device))

    model.to(device).eval()
    print("Model successfully loaded and set to evaluation mode.\n")
    return model

# Load the trained model
model_path = "C:\\Users\\jayve\\solar_defect_fully_trained_model.pth"
model = load_trained_model(model_path, num_classes)

# Path to test images
test_image_dir = "D:\\projects\\solar_defect\\dataset\\test"

# Define image transformations
transform = T.Compose([T.ToTensor()])

# Function to run inference on an image
def predict_image(image_path, model, threshold=0.5):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_tensor = transform(image).to(device)

    # Run inference
    with torch.no_grad():
        prediction = model([image_tensor])[0]

    # Process predictions
    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()
    labels = prediction['labels'].cpu().numpy()

    detected_objects = []
    for box, score, label in zip(boxes, scores, labels):
        if score >= threshold:
            detected_objects.append((CLASS_NAMES[label - 1], box, score))

    return image, detected_objects

# Visualize results
def visualize_predictions(image, predictions, image_name):
    for label, box, score in predictions:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, f"{label} ({score:.2f})", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.title(image_name)
    plt.axis("off")
    plt.show()

# Run inference on the first 50 test images
test_images = sorted(os.listdir(test_image_dir))[:50]  # Take first 50 images

# Process images one by one
defect_counts = {}
for img_name in test_images:
    img_path = os.path.join(test_image_dir, img_name)
    print(f"Processing: {img_name}")

    # Predict
    image, predictions = predict_image(img_path, model)

    # Count defects
    for label, _, _ in predictions:
        defect_counts[label] = defect_counts.get(label, 0) + 1

    # Visualize
    visualize_predictions(image, predictions, img_name)

# Print defect summary
print("\nDefect Summary:")
for defect, count in defect_counts.items():
    print(f"{defect}: {count}")


cpu
Loading Faster R-CNN model...
Loading trained weights from: C:\Users\jayve\solar_defect_fully_trained_model.pth
Model successfully loaded and set to evaluation mode.

Processing: img004602.jpg


In [1]:
import torch
import torchvision.models.detection as detection

device = torch.device("cpu")  # Force CPU
model = detection.fasterrcnn_resnet50_fpn(weights=None).to(device)
print("Model loaded successfully")

Model loaded successfully
